In [2]:
!pip install librosa scikit-learn joblib soundfile numpy matplotlib


In [ ]:
import os
import librosa
import numpy as np

def extract_features(file_path, sr=16000, duration=3):
    try:
        y, _ = librosa.load(file_path, sr=sr, duration=duration)

        # Pad if too short
        if len(y) < 2048:
            padding = 2048 - len(y)
            y = np.pad(y, (0, padding), mode='constant')

        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
        return np.mean(mfcc.T, axis=0)
    except Exception as e:
        print(f" Error: {file_path} - {e}")
        return None


def load_dataset(base_path):
    X, y = [], []
    for label in os.listdir(base_path):
        label_path = os.path.join(base_path, label)
        if not os.path.isdir(label_path):
            continue
        for file in os.listdir(label_path):
            file_path = os.path.join(label_path, file)
            if file.lower().endswith(".wav"):
                features = extract_features(file_path)
                if features is not None:
                    X.append(features)
                    y.append(1 if label.lower() == "fake" else 0)
    print(f" Total files loaded: {len(X)}")
    return np.array(X), np.array(y)


dataset_path = r"D:\archive\for-norm\for-norm\training"

X, y = load_dataset(dataset_path)
print("X shape:", X.shape)
print("y shape:", y.shape)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Predict & Evaluate
y_pred = model.predict(X_test)
print(" Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


In [6]:
import joblib
joblib.dump(model, "deepfake_audio_detector.pkl")


['deepfake_audio_detector.pkl']

In [7]:
!pip install gradio


  Using cached fsspec-2025.5.1-py3-none-any.whl.metadata (11 kB)
  Using cached filelock-3.18.0-py3-none-any.whl.metadata (2.9 kB)
   ---------------------------------------- 0.0/54.2 MB ? eta -:--:--
   - -------------------------------------- 1.6/54.2 MB 9.4 MB/s eta 0:00:06
   -- ------------------------------------- 3.4/54.2 MB 8.8 MB/s eta 0:00:06
   ---- ----------------------------------- 5.5/54.2 MB 9.6 MB/s eta 0:00:06
   ----- ---------------------------------- 7.6/54.2 MB 10.0 MB/s eta 0:00:05
   ------- -------------------------------- 10.2/54.2 MB 10.3 MB/s eta 0:00:05
   --------- ------------------------------ 12.6/54.2 MB 10.5 MB/s eta 0:00:04
   ---------- ----------------------------- 14.4/54.2 MB 10.3 MB/s eta 0:00:04
   ---------- ----------------------------- 14.7/54.2 MB 9.9 MB/s eta 0:00:05
   ------------ --------------------------- 16.5/54.2 MB 9.0 MB/s eta 0:00:05
   ------------- -------------------------- 18.1/54.2 MB 8.9 MB/s eta 0:00:05
   -------------- -

In [4]:
import gradio as gr
import librosa
import numpy as np
import joblib

# Load your model
model = joblib.load("deepfake_audio_detector.pkl")

def extract_features_for_single_file(file_path):
    try:
        y, sr = librosa.load(file_path, sr=16000)
        if len(y) < 2048:
            y = np.pad(y, (0, 2048 - len(y)), mode='constant')
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
        mfcc_mean = np.mean(mfcc, axis=1)
        return mfcc_mean
    except Exception as e:
        print("Error:", e)
        return None

def predict(audio):
    features = extract_features_for_single_file(audio)
    if features is None:
        return "❌ Error processing audio"
    prediction = model.predict([features])[0]
    return "🧑‍🎤 Real Voice" if prediction == 0 else "🎭 Deepfake Voice"

gr.Interface(
    fn=predict,
    inputs=gr.Audio(type="filepath", label="Upload or Record Your Voice"),
    outputs=gr.Text(label="Prediction"),
    title="🎙️ Deepfake Voice Detector",
    description="Upload or record a voice clip to check if it's real or deepfake."
).launch()


* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.
